# Stage 03 — UIO Fleet Projection & Demand
**Dashboard pages:** UIO Snapshot · UIO Forecast & Demand
**Tabs:** UIO by Model Historical · UIO Forecast · UIO-Based Demand

**Stock-flow model:** UIO(t) = UIO(t-1) + new_sales(t) - UIO(t-1) × 0.00427  (5% annual attrition)

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
uio    = load("uio.parquet")
uio_fc = load("uio_forecast.parquet")
uio_dm = load("uio_based_demand.parquet")

print("UIO historical  :", uio.shape, "| cols:", uio.columns.tolist())
print("UIO forecast    :", uio_fc.shape, "| cols:", uio_fc.columns.tolist())
print("UIO-based demand:", uio_dm.shape, "| cols:", uio_dm.columns.tolist())


## UIO by Model — Historical Fleet

In [ ]:
# uio.parquet has columns: Model, 2013, 2013(20%), 2014, 2014(20%), ...
# year columns are the running fleet count; (20%) columns are 80% attrition-adjusted UIO
year_cols = [c for c in uio.columns if c != "Model" and not c.endswith("%)")]
adj_cols  = [c for c in uio.columns if c.endswith("%)")]

if "Model" in uio.columns and year_cols:
    uio_pivot = uio.set_index("Model")[year_cols].T
    uio_pivot.index = pd.to_numeric(uio_pivot.index, errors="coerce")
    uio_pivot = uio_pivot.sort_index()

    fig, axes = plt.subplots(1,2,figsize=(15,5))
    uio_pivot.plot(ax=axes[0], color=PALETTE[:len(uio_pivot.columns)], lw=2, marker="o", ms=4)
    axes[0].set_title("UIO by Model — Historical Fleet (raw cumulative)")
    axes[0].set_xlabel("Year"); axes[0].set_ylabel("Units in Operation")
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{int(x):,}"))
    axes[0].legend(title="Model",fontsize=8)

    # Stacked bar latest year
    latest = uio.set_index("Model")[year_cols[-1]].sort_values(ascending=False)
    latest.plot(kind="bar",ax=axes[1],color=PALETTE[:len(latest)],edgecolor="white")
    axes[1].set_title(f"UIO by Model — Latest Year ({year_cols[-1]})")
    axes[1].set_ylabel("Units in Operation"); axes[1].tick_params(axis="x",rotation=45)
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{int(x):,}"))
    for bar,val in zip(axes[1].patches,latest.values):
        axes[1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+20,f"{val:,}",ha="center",fontsize=8)
    plt.tight_layout(); plt.show()
    print(uio.set_index("Model")[year_cols].to_string())


## UIO Forecast — 36-Month Projection (Stock-Flow)

In [ ]:
date_col  = next((c for c in ["month","date","ds","period"] if c in uio_fc.columns),None)
uio_col   = next((c for c in ["total_uio","uio","fleet_size"] if c in uio_fc.columns),None)
model_col = next((c for c in ["model","Model"] if c in uio_fc.columns),None)

if date_col and uio_col:
    if model_col:
        # Per-model forecast lines
        fig,axes = plt.subplots(1,2,figsize=(15,5))
        for i,mdl in enumerate(uio_fc[model_col].unique()):
            d = uio_fc[uio_fc[model_col]==mdl].sort_values(date_col)
            axes[0].plot(range(len(d)),d[uio_col],color=PALETTE[i%len(PALETTE)],lw=2,label=mdl)
        axes[0].set_title("UIO Forecast by Model — 36-Month Projection")
        axes[0].set_ylabel("Projected UIO"); axes[0].legend(fontsize=8)
        axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{int(x):,}"))
        # Total UIO
        total = uio_fc.groupby(date_col)[uio_col].sum().reset_index().sort_values(date_col)
        axes[1].plot(range(len(total)),total[uio_col],color=PALETTE[0],lw=2.5)
        axes[1].set_title("Total Fleet Projection (all models)
"
                          "Stock-flow: UIO(t) = UIO(t-1) + sales(t) - UIO(t-1)*0.00427")
        axes[1].set_ylabel("Total UIO")
        axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{int(x):,}"))
        plt.tight_layout(); plt.show()
    else:
        uio_fc_sorted = uio_fc.sort_values(date_col)
        fig,ax = plt.subplots(figsize=(13,4))
        ax.plot(range(len(uio_fc_sorted)),uio_fc_sorted[uio_col],color=PALETTE[0],lw=2.5)
        ax.set_title("Total UIO Forecast — 36-Month Projection")
        ax.set_ylabel("UIO"); plt.tight_layout(); plt.show()
else:
    print("UIO forecast columns:", uio_fc.columns.tolist())
    print(uio_fc.head())


In [ ]:
# Attrition sensitivity
r_annual = 0.05; r_monthly = 1-(1-r_annual)**(1/12)
print(f"Annual attrition rate    : {r_annual*100:.1f}%")
print(f"Monthly attrition rate   : {r_monthly*100:.4f}%")
print(f"5-year survival (80% UIO): {(1-r_annual)**5*100:.1f}%")
print()
print("Sensitivity:")
for rate in [0.04,0.05,0.06,0.07]:
    r_m = 1-(1-rate)**(1/12)
    print(f"  Annual {rate*100:.0f}%  -> monthly {r_m*100:.4f}%  -> 5yr survival {(1-rate)**5*100:.1f}%")


## UIO-Based Demand (UIO Forecast → Spare Parts Demand)

In [ ]:
print(f"UIO-based demand: {len(uio_dm):,} parts")
print(uio_dm.head(10).to_string())
print()

fig,axes = plt.subplots(1,2,figsize=(13,5))
if "uio_demand_leadtime" in uio_dm.columns:
    d = uio_dm["uio_demand_leadtime"].clip(upper=uio_dm["uio_demand_leadtime"].quantile(0.95))
    axes[0].hist(d.dropna(),bins=60,color=PALETTE[4],edgecolor="white",alpha=0.8)
    axes[0].set_title("UIO-Based 3-Month Demand Distribution
"
                      "(replacement_freq × projected_UIO × 0.60 / 12 × 3)")
    axes[0].set_xlabel("Units (3-month lead-time)"); axes[0].set_ylabel("SKUs")

if "model_uio_total" in uio_dm.columns and "compatible_models" in uio_dm.columns:
    top_demand = uio_dm.nlargest(15,"uio_demand_leadtime")
    top_demand["label"] = top_demand["description"].str[:35]
    top_demand.set_index("label")["uio_demand_leadtime"].sort_values().plot(
        kind="barh",ax=axes[1],color=PALETTE[0],edgecolor="white")
    axes[1].set_title("Top 15 Parts by UIO-Based Lead-Time Demand")
    axes[1].set_xlabel("Projected 3-month demand (units)")
plt.tight_layout(); plt.show()

print("
Top 10 by UIO-based demand:")
print(uio_dm.nlargest(10,"uio_demand_leadtime")[
    ["material_9","description","compatible_models","uio_demand_leadtime"]].to_string(index=False))


**How UIO demand feeds the order plan:** `uio_demand_leadtime` per part is used in the UIO-Based Plan tab of the Order Plan page as an alternative demand signal — particularly important for recently launched models where historical issue data is sparse.